# Part 5：模型封裝、Bootstrap 與 Slice 分析

## 研究問題

如何把 marker-aligned compact model 工程化，並量化不確定性與 workload slices？

## 目前結論

Part 5 封裝 Part 3 模型、bootstrap interval、feature stability 與 slice metrics；定位仍是 marker-aligned oracle model。

> 本 Notebook 新增於既有研究產物之外，不會取代原始 Python scripts 或 CSV／Markdown 報告。

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display


def find_project_root() -> Path:
    """Find the repository whether Jupyter starts at the repo or notebook folder."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "analysis").is_dir() and (candidate / "analysis_outputs").is_dir():
            return candidate
    raise FileNotFoundError("找不到專案根目錄；請從本 repository 內啟動 Jupyter。")


PROJECT_ROOT = find_project_root()
OUTPUT_ROOT = PROJECT_ROOT / "analysis_outputs"
RUN_EXPENSIVE = False  # 明確改成 True 才會重跑分析腳本。


def project_path(relative: str) -> Path:
    return PROJECT_ROOT / relative


def load_csv(relative: str, **kwargs) -> pd.DataFrame:
    path = project_path(relative)
    if not path.exists():
        raise FileNotFoundError(f"缺少資料：{path}")
    return pd.read_csv(path, **kwargs)


def show_markdown(relative: str) -> None:
    path = project_path(relative)
    display(Markdown(path.read_text(encoding="utf-8")))


def inspect_inputs(paths) -> pd.DataFrame:
    rows = []
    for relative in paths:
        path = project_path(relative)
        rows.append({
            "path": relative,
            "exists": path.exists(),
            "type": "directory" if path.is_dir() else "file",
            "size_mib": round(path.stat().st_size / 1024**2, 3) if path.is_file() else None,
        })
    return pd.DataFrame(rows)


def run_commands(commands) -> None:
    """Show commands by default; run only after explicit opt-in."""
    for args in commands:
        command = [sys.executable, *args]
        print(" ".join(map(str, command)))
        if RUN_EXPENSIVE:
            subprocess.run(command, cwd=PROJECT_ROOT, check=True)


print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"RUN_EXPENSIVE = {RUN_EXPENSIVE}")

## 實驗步驟

1. 固定 Part 3 compact feature contract。
2. 比較 Part 1–3 階段結果。
3. 對 OOF errors 做 bootstrap uncertainty。
4. 依 workload slice 檢查誤差。
5. 訓練方向別 deployable bundles 並輸出 manifest。

## 輸入與 provenance

下列 cell 只檢查輸入是否存在，不修改資料。

In [ ]:
INPUTS = [
    "analysis_outputs/model_experiments/part3_profile_models/metrics.csv",
    "analysis_outputs/model_experiments/part3_profile_models/predictions.csv",
    "analysis_outputs/cross_device_samples.csv",
]
display(inspect_inputs(INPUTS))

## 重現入口

下列命令依現有 pipeline 排列。預設只列印；將 RUN_EXPENSIVE 明確改成 True 才會執行。

In [ ]:
COMMANDS = [
    ["analysis/finalize_model_experiment.py"],
]
run_commands(COMMANDS)

## 結果表

In [ ]:
METRICS_PATH = "analysis_outputs/model_experiments/part5_final/stage_comparison.csv"

metrics = load_csv(METRICS_PATH)
sort_column = "mape_percent" if "mape_percent" in metrics.columns else "mean_period_ape_percent"
display(metrics.sort_values(sort_column).reset_index(drop=True))

if "direction" in metrics.columns and "model" in metrics.columns:
    plot_data = metrics.sort_values(sort_column).groupby("direction", as_index=False).first()
    ax = plot_data.plot.bar(x="direction", y=sort_column, legend=False, figsize=(8, 4))
    ax.set_ylabel(sort_column)
    ax.set_title("各方向最佳列（依表內指標排序）")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
for relative in [
    "analysis_outputs/model_experiments/part5_final/bootstrap_intervals.csv",
    "analysis_outputs/model_experiments/part5_final/slice_metrics.csv",
    "analysis_outputs/model_experiments/part5_final/feature_stability.csv",
    "analysis_outputs/model_experiments/part5_final/final_alpha_selection.csv",
]:
    print(relative)
    display(load_csv(relative))

## 既有最終報告快照

> 來源：`analysis_outputs/model_experiments/part5_final/final_report.md`。這是建立 Notebook 時嵌入的結果快照；重新執行分析後，請以磁碟上的最新報告為準。

# Final model experiment report

## Decision

Use the 15-feature compact GPU Ridge model separately for RTX4090-to-RTX5090 and RTX5090-to-RTX4090. It is deployment-aligned: source profiling and workload fields are inputs; target unprofiled baseline runtime is label-only.

## Stage comparison

| Direction | Stage | Deployment eligible | Best model | MAE ms | MAPE | Max APE | R2 |
|---|---|---|---|---:|---:|---:|---:|
| RTX4090_to_RTX5090 | part1_idealized | no | log_source_ols | 1.835 | 4.10% | 11.97% | 0.9852 |
| RTX4090_to_RTX5090 | part2_idealized | no | huber_interactions_log_target | 1.117 | 3.11% | 11.29% | 0.9930 |
| RTX4090_to_RTX5090 | part3_deployment | yes | compact_gpu_ridge | 1.284 | 3.14% | 10.74% | 0.9935 |
| RTX4090_to_RTX5090 | part4_deployment | yes | static_compact_gpu_ridge | 1.284 | 3.14% | 10.74% | 0.9935 |
| RTX5090_to_RTX4090 | part1_idealized | no | global_ratio_median | 2.493 | 4.23% | 12.50% | 0.9702 |
| RTX5090_to_RTX4090 | part2_idealized | no | ridge_main_log_target | 2.046 | 4.27% | 12.92% | 0.9864 |
| RTX5090_to_RTX4090 | part3_deployment | yes | compact_gpu_ridge | 2.028 | 4.26% | 10.22% | 0.9900 |
| RTX5090_to_RTX4090 | part4_deployment | yes | static_compact_gpu_ridge | 2.028 | 4.26% | 10.22% | 0.9900 |

## Workload-bootstrap uncertainty for the selected deployment model

Intervals resample the 24 held-out workload predictions within each direction. They quantify workload-sampling uncertainty for this GPU pair, not unseen-device uncertainty.

| Direction | Metric | Estimate | 95% bootstrap interval |
|---|---|---:|---:|
| RTX4090_to_RTX5090 | mae_ms | 1.284 ms | 0.962 to 1.651 ms |
| RTX4090_to_RTX5090 | rmse_ms | 1.549 ms | 1.101 to 1.965 ms |
| RTX4090_to_RTX5090 | mape_percent | 3.138% | 2.299 to 4.145% |
| RTX5090_to_RTX4090 | mae_ms | 2.028 ms | 1.526 to 2.605 ms |
| RTX5090_to_RTX4090 | rmse_ms | 2.429 ms | 1.788 to 3.031 ms |
| RTX5090_to_RTX4090 | mape_percent | 4.257% | 3.263 to 5.332% |

## Error slices

### RTX4090_to_RTX5090

| Slice | Value | N | MAE ms | MAPE | Bias ms | Max APE |
|---|---|---:|---:|---:|---:|---:|
| batch | 4 | 6 | 1.449 | 4.79% | +0.212 | 10.74% |
| batch | 8 | 6 | 1.033 | 3.05% | +0.185 | 6.18% |
| batch | 16 | 6 | 0.763 | 1.73% | -0.079 | 2.19% |
| batch | 32 | 6 | 1.890 | 2.98% | -0.737 | 6.78% |
| imgsz | 320 | 8 | 1.520 | 4.69% | +0.030 | 10.74% |
| imgsz | 480 | 8 | 0.838 | 1.97% | -0.124 | 2.60% |
| imgsz | 640 | 8 | 1.493 | 2.76% | -0.221 | 4.84% |
| amp | False | 12 | 1.435 | 3.50% | -0.181 | 10.74% |
| amp | True | 12 | 1.132 | 2.77% | -0.029 | 6.78% |

### RTX5090_to_RTX4090

| Slice | Value | N | MAE ms | MAPE | Bias ms | Max APE |
|---|---|---:|---:|---:|---:|---:|
| batch | 4 | 6 | 2.081 | 6.10% | +0.262 | 9.06% |
| batch | 8 | 6 | 2.224 | 5.43% | -0.211 | 10.22% |
| batch | 16 | 6 | 1.067 | 2.23% | -0.253 | 3.12% |
| batch | 32 | 6 | 2.741 | 3.28% | +1.794 | 5.08% |
| imgsz | 320 | 8 | 1.096 | 3.06% | -0.057 | 5.63% |
| imgsz | 480 | 8 | 1.977 | 4.45% | +0.590 | 9.06% |
| imgsz | 640 | 8 | 3.013 | 5.26% | +0.661 | 10.22% |
| amp | False | 12 | 2.061 | 4.27% | +0.482 | 10.22% |
| amp | True | 12 | 1.996 | 4.25% | +0.314 | 9.06% |

## Interpretation and limits

- Against the best Part 2 idealized model, the deployment model changes mean APE by only +0.026 percentage points for RTX4090-to-RTX5090 (paired Wilcoxon p=0.584) and -0.014 points for RTX5090-to-RTX4090 (p=0.922); no measurable accuracy penalty is evident in this dataset.
- Compact GPU metrics capture most useful profiling signal; adding full GPU summaries, trace, or current HPCT artifact proxies does not improve nested-LOOW error.
- Current evaluation supports unseen-workload generalization only for the known RTX4090/RTX5090 directed pair.
- Hardware descriptors are constant within each direction and cannot establish unseen-device generalization with only two GPU models.
- Current source profiling covers 108 steady iterations. Shorter 5/10/20-iteration window accuracy remains unmeasured.
- RTX4090 and RTX5090 use different CUDA trace backends, so trace features require additional harmonization before treating them as device-independent.
- Current HPCToolkit features are quality/complexity proxies. Numeric CCT attribution must be extracted before HPCToolkit can be judged as a performance-feature source.

## Recommended next evidence

1. Re-extract Nsight metrics from progressively shorter profiling windows and plot accuracy versus profiling cost.
2. Add at least one third GPU and evaluate leave-one-device-out; until then, avoid claims about arbitrary unseen GPU hardware.
3. Add more model architectures and data-pipeline conditions so workload generalization is not limited to YOLO26n and the current batch/image/AMP grid.
4. Extract numeric HPCToolkit CCT metrics only if kernel/calling-context attribution can be validated against Nsight.

## Packaged models

- `RTX4090_to_RTX5090`: `models\RTX4090_to_RTX5090_compact_gpu_ridge.joblib`, alpha=0.1, 15 inputs.
- `RTX5090_to_RTX4090`: `models\RTX5090_to_RTX4090_compact_gpu_ridge.joblib`, alpha=0.1, 15 inputs.


## 解讀檢查清單

- 區分 OOF evaluation 與重新用全部資料訓練的 production bundle。
- 不把 target runtime、oracle marker 或 future information 當作 deployment feature。
- 報告方向別結果、combined summary 與 worst cases。
- 結論只適用於目前資料、GPU 節點與 workload family。